# Final results: `dist_bootHL` (dist head + bootstrap HL, 3-seed ensemble)

Out-of-fold (OOF) test results of the winner over the 5 CV folds: every subject is tested exactly once, by the
3 models of the fold whose test group it is in (they never saw that subject). 47 subjects: 20 multisite-ppg, 27 wrist-finger-ppg.

**Shipped pipeline** (what `estimate_heartrate` runs): 10 s windows every 2 s -> 3-seed ensemble -> input gate
(dist curve std above the 90 % val quantile = no weight) -> Kalman + RTS (R = (c · score)², q = (m · step std)²).
The interface must return a value at every time point, so there is **no output gate**: gated windows are bridged by
the smoother. Per fold the smoother parameters and the gate limit are the ones `smooth_report.py` tuned on that
fold's val group.

Two scopes:
- **covered**: windows `smooth_report.py` scores (gated runs <= 30 s bridged; longer runs left out). Reproduces its CSV.
- **all**: every labelled window, as the shipped function returns them (also long gated runs; a segment with no
  passing window falls back to the raw ensemble value).

Labels: mean ECG-derived HR over each 10 s window. All labelled windows of the test recordings (not capped).

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from bpm.aggregate import BIN_EDGES, decide, ltgo_folds, macro, parse_tag, subject_mae

PREDS = ROOT / "cache" / "preds"
RESULTS = ROOT / "cache" / "results"
OUT = ROOT / "cache" / "deck" / "cv_story"   # figures + values.json for the deck
OUT.mkdir(parents=True, exist_ok=True)
DATASETS = ["multisite-ppg", "wrist-finger-ppg"]

In [2]:
from bpm.quality import balanced_mae, fit_limit
from bpm.smoothing import segments
from bpm.split import fold_split, load_folds
from smooth_report import FOLDS_PATH, smooth, train_q

CFG = "dist_bootHL"
OUT = ROOT / "cache" / "plots" / "final"   # figures + values.json of this notebook (save()/put() write here)
OUT.mkdir(parents=True, exist_ok=True)
VALUES_PATH, VALUES = OUT / "values.json", {}

## Style

Single source of truth for every plot below. Colors come from a colorblind-safe palette and are bound to **roles**: a config keeps its color in every figure of both notebooks. Plot cells never use a raw hex color or font size.

In [3]:
# palette of docs/hr_cnn_bilstm_results.odp (the deck these figures get merged into):
# ink #212a30, accent green #1e7f5c, red #b03a2e, secondary #555b60, muted #7a7f84, light #c9d1cd, card #edf2f0
COLORS = {
    # configs (role = model family): baselines slate/grey, MSE + boot red, dist + boot green
    "mse": "#212a30", "huber10": "#555b60", "mae": "#7a7f84",
    "mse_bootHL": "#b03a2e", "mse_bootHL_lds0.5": "#b03a2e", "mse_bootHL_lds1": "#b03a2e",
    "dist_bootHL": "#1e7f5c", "dist_bootHL_lds0.5": "#1e7f5c", "dist_bootHL_lds1": "#1e7f5c",
    # datasets
    "multisite-ppg": "#212a30", "wrist-finger-ppg": "#1e7f5c",
    # scores (line style separates the greys, see SCORE_LS)
    "score_std": "#1e7f5c", "score_mc": "#b03a2e", "score_spread": "#212a30",
    "prominence": "#7a7f84", "skew": "#7a7f84",
    # smoothers
    "raw": "#7a7f84", "median": "#555b60", "kalman": "#1e7f5c", "hmm": "#212a30",
    # signals
    "truth": "#212a30", "ideal": "#7a7f84", "bad": "#b03a2e", "good": "#1e7f5c", "neutral": "#c9d1cd",
    # splits (train vs test): test carries the accent
    "train": "#212a30", "val": "#7a7f84", "test": "#1e7f5c",
    # ink and surface
    "text": "#212a30", "muted": "#7a7f84", "grid": "#e8ecea", "surface": "#ffffff",
}
SCORE_LS = {"score_std": "-", "score_mc": "-", "score_spread": "-", "prominence": "-", "skew": "--"}
# LDS power shown by marker (scatter) / line style (curves), so 3 hues cover all 9 configs
MARKERS = {"": "o", "lds0.5": "s", "lds1": "^"}
LINESTYLES = {"": "-", "lds0.5": "--", "lds1": ":"}
LABELS = {
    "mse": "MSE", "huber10": "Huber δ10", "mae": "MAE",
    "mse_bootHL": "MSE + boot", "mse_bootHL_lds0.5": "MSE + boot + LDS 0.5", "mse_bootHL_lds1": "MSE + boot + LDS 1",
    "dist_bootHL": "dist + boot", "dist_bootHL_lds0.5": "dist + boot + LDS 0.5", "dist_bootHL_lds1": "dist + boot + LDS 1",
}
STYLE = {"fig_w": 10, "row_h": 4.2, "dpi": 110, "save_dpi": 200, "font": 11, "title": 13,
         "lw": 1.8, "lw_thin": 1.0, "marker": 70, "alpha": 0.25}

plt.rcParams.update({
    "figure.dpi": STYLE["dpi"], "figure.facecolor": COLORS["surface"], "axes.facecolor": COLORS["surface"],
    "figure.constrained_layout.use": True,
    "font.size": STYLE["font"], "axes.titlesize": STYLE["title"], "axes.labelsize": STYLE["font"],
    "legend.fontsize": STYLE["font"] - 1, "legend.frameon": False,
    "text.color": COLORS["text"], "axes.labelcolor": COLORS["text"], "axes.edgecolor": COLORS["muted"],
    "xtick.color": COLORS["muted"], "ytick.color": COLORS["muted"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": COLORS["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": STYLE["lw"],
    "font.family": "Lato",  # the deck's body font as rendered here (Calibri falls back to it)
})
from matplotlib.colors import LinearSegmentedColormap
DENSITY_CMAP = LinearSegmentedColormap.from_list("deck", ["#edf2f0", "#1e7f5c", "#212a30"])


def figsize(rows=1, w=None):
    return (w or STYLE["fig_w"], STYLE["row_h"] * rows)


def lds_of(cfg):
    return next((p for p in ("lds0.5", "lds1") if cfg.endswith(p)), "")


def save(fig, name):
    """Figure for the deck: cache/deck/cv_story/<name>.png."""
    fig.savefig(OUT / f"{name}.png", dpi=STYLE["save_dpi"], facecolor=COLORS["surface"])


VALUES_PATH = OUT / "values.json"
VALUES = json.loads(VALUES_PATH.read_text()) if VALUES_PATH.exists() else {}


def put(section, **kv):
    """Record numbers the deck cites (values.json, merged across both notebooks)."""
    conv = lambda v: v.item() if hasattr(v, "item") else v
    VALUES.setdefault(section, {}).update({k: conv(v) for k, v in kv.items()})
    VALUES_PATH.write_text(json.dumps(VALUES, indent=1, default=float))


BIN_LABELS = [f"{lo}-{hi}" if np.isfinite(hi) else f"{lo}+" for lo, hi in zip(BIN_EDGES[:-1], BIN_EDGES[1:])]


def bin_of(truth):
    return pd.cut(truth, BIN_EDGES, right=False, labels=BIN_LABELS)

## Re-run the shipped pipeline per fold

Same functions and tuned parameters as `smooth_report.py` (row `kalman`, input gate `coverage`, output gate `none`).
Check: the covered subject-macro MAE per fold equals the CSV value.

In [4]:
F = pd.concat([pd.read_csv(RESULTS / f"smooth_ens_{CFG}_fold{k}.csv") for k in range(5)], ignore_index=True)
groups, _ = load_folds(FOLDS_PATH)
rows, PARAMS = [], {}
for k in range(5):
    row = F[(F["fold"] == k) & (F["smoother"] == "kalman") & (F["in_gate"] == "coverage") & (F["out_gate"] == "none")]
    assert len(row) == 1, (k, len(row))
    row = row.iloc[0]
    params = eval(row["params"])  # dict literal written by smooth_report
    z = np.load(PREDS / f"ens_{CFG}_fold{k}.npz")
    v = z["split"] == "val"
    in_limit = fit_limit(z["scores"][v], np.abs(z["pred"] - z["truth"])[v], z["subject"][v], z["dataset_source"][v])[0]
    q = train_q(set(fold_split(groups, k)["train"]))
    for p in sorted((ROOT / "cache" / "sequences" / f"ens_{CFG}_fold{k}").glob("*.npz")):
        zz = np.load(p)
        s = {key: (zz[key].item() if zz[key].ndim == 0 else zz[key]) for key in zz.files}
        if s["split"] != "test":
            continue
        out, _, cov = smooth(s, "kalman", params, in_limit, q, None, 30)
        raw = s["bpm"].astype(float)
        ok = ~(s["score"].astype(float) > in_limit)
        shipped = out.copy()
        for sl in segments(s["idx"]):
            if not ok[sl].any():  # nothing to bridge from: Kalman has no measurement, fall back to raw
                shipped[sl] = raw[sl]
        lab = np.isfinite(s["label"])
        rows.append(pd.DataFrame({
            "fold": k, "recording": p.stem, "subject": s["subject"], "dataset": s["dataset_source"],
            "t_s": s["idx"][lab] * 2, "truth": s["label"][lab].astype(float), "raw": raw[lab],
            "kalman": shipped[lab], "covered": cov[lab], "gated": ~ok[lab]}))
    PARAMS[k] = {**params, "in_limit": in_limit, **{f"q_{d}": v for d, v in q.items()}, "csv_mae": row["mae_macro"]}
W = pd.concat(rows, ignore_index=True)
for k, g in W.groupby("fold"):
    c = g["covered"].to_numpy()
    mae = balanced_mae(np.abs(g["kalman"] - g["truth"]).to_numpy(), g["subject"].to_numpy(), g["dataset"].to_numpy(), c)[0]
    assert abs(mae - PARAMS[k]["csv_mae"]) < 1e-6, (k, mae, PARAMS[k]["csv_mae"])
print(len(W), "labelled test windows,", W["subject"].nunique(), "subjects; covered", f"{W['covered'].mean():.1%}",
      "| gated by input gate", f"{W['gated'].mean():.1%}")
pd.DataFrame(PARAMS).T.round(3)

q from 362 fold-train HR curves: multisite-ppg 1.006, wrist-finger-ppg 1.062


q from 419 fold-train HR curves: multisite-ppg 1.191, wrist-finger-ppg 0.997


q from 461 fold-train HR curves: multisite-ppg 1.112, wrist-finger-ppg 1.080


q from 440 fold-train HR curves: multisite-ppg 0.932, wrist-finger-ppg 1.063


q from 370 fold-train HR curves: multisite-ppg 0.990, wrist-finger-ppg 1.109


427994 labelled test windows, 47 subjects; covered 98.5% | gated by input gate 14.5%


,m,c,in_limit,q_multisite-ppg,q_wrist-finger-ppg,csv_mae
0,2.0,0.5,16.205,1.006,1.062,3.522
1,1.0,0.5,15.833,1.191,0.997,3.308
2,1.0,1.0,18.199,1.112,1.080,3.577
3,1.0,0.5,18.254,0.932,1.063,3.529
4,4.0,4.0,15.087,0.990,1.109,3.890


## Headline table

Per sub-dataset and both (mean over the 2 sub-datasets, so the 37k multisite windows do not dominate).
`mae_subject` = mean of per-subject MAE (the CV ranking metric); `mae_window`, `rmse`, `bias` (pred - truth),
Pearson `r`, `within_5` / `within_10` = share of windows with |error| <= 5 / 10 BPM.

In [5]:
def summarize(d, col):
    e = d[col] - d["truth"]
    return {"subjects": d["subject"].nunique(), "windows": len(d),
            "mae_subject": d.assign(err=e.abs()).groupby("subject")["err"].mean().mean(),
            "mae_window": e.abs().mean(), "rmse": np.sqrt((e ** 2).mean()), "bias": e.mean(),
            "r": np.corrcoef(d[col], d["truth"])[0, 1],
            "within_5": (e.abs() <= 5).mean(), "within_10": (e.abs() <= 10).mean()}


PIPES = {"raw ensemble": ("raw", None), "shipped, all windows": ("kalman", None), "shipped, covered": ("kalman", "covered")}
T = []
for name, (col, mask) in PIPES.items():
    d = W[W[mask]] if mask else W
    per = {ds: summarize(d[d["dataset"] == ds], col) for ds in DATASETS}
    both = {m: np.mean([per[ds][m] for ds in DATASETS]) for m in per[DATASETS[0]]}
    both.update(subjects=d["subject"].nunique(), windows=len(d))
    for ds, m in [*per.items(), ("both", both)]:
        T.append({"pipeline": name, "dataset": ds, "coverage": len(d[d["dataset"] == ds]) / len(W[W["dataset"] == ds])
                  if ds != "both" else len(d) / len(W), **m})
T = pd.DataFrame(T).set_index(["pipeline", "dataset"])
T.to_csv(RESULTS / f"final_{CFG}.csv")
put("table", **{f"{p}|{d}|{m}": T.loc[(p, d), m] for p, d in T.index for m in T.columns})
T.round(3)

coverage  subjects  windows  \
pipeline             dataset                                         
raw ensemble         multisite-ppg        1.000        20   413798   
                     wrist-finger-ppg     1.000        27    14196   
                     both                 1.000        47   427994   
shipped, all windows multisite-ppg        1.000        20   413798   
                     wrist-finger-ppg     1.000        27    14196   
                     both                 1.000        47   427994   
shipped, covered     multisite-ppg        0.984        20   407189   
                     wrist-finger-ppg     1.000        27    14196   
                     both                 0.985        47   421385   

                                       mae_subject  mae_window    rmse   bias  \
pipeline             dataset                                                    
raw ensemble         multisite-ppg           6.735       7.487  13.350  1.806   
                     wrist-finger-ppg        1.477       1.498   2.531  0.165   
                     both                    4.106       4.493   7.941  0.985   
shipped, all windows multisite-ppg           5.815       6.465  10.845  1.216   
                     wrist-finger-ppg        1.466       1.478   2.236  0.121   
                     both                    3.641       3.971   6.541  0.669   
shipped, covered     multisite-ppg           5.657       6.276  10.542  1.167   
                     wrist-finger-ppg        1.466       1.478   2.236  0.121   
                     both                    3.562       3.877   6.389  0.644   

                                           r  within_5  within_10  
pipeline             dataset                                       
raw ensemble         multisite-ppg     0.574     0.619      0.745  
                     wrist-finger-ppg  0.970     0.950      0.991  
                     both              0.772     0.785      0.868  
shipped, all windows multisite-ppg     0.717     0.632      0.774  
                     wrist-finger-ppg  0.977     0.959      0.995  
                     both              0.847     0.796      0.885  
shipped, covered     multisite-ppg     0.725     0.639      0.781  
                     wrist-finger-ppg  0.977     0.959      0.995  
                     both              0.851     0.799      0.888

## Predicted vs ECG

Density of all test windows (shipped pipeline, all windows). The diagonal is perfect agreement; the dashed line is
the least-squares fit (a slope below 1 = pull towards the middle).

In [6]:
fig, axes = plt.subplots(1, 2, figsize=figsize(1.35), sharex=True, sharey=True)
lim = (35, 175)
for ax, ds in zip(axes, DATASETS):
    d = W[W["dataset"] == ds]
    hb = ax.hexbin(d["truth"], d["kalman"], gridsize=60, extent=(*lim, *lim), bins="log", cmap=DENSITY_CMAP, mincnt=1)
    ax.plot(lim, lim, color=COLORS["ideal"], lw=STYLE["lw_thin"])
    slope, icpt = np.polyfit(d["truth"], d["kalman"], 1)
    ax.plot(lim, np.polyval([slope, icpt], lim), color=COLORS["bad"], ls="--", lw=STYLE["lw_thin"],
            label=f"fit: slope {slope:.2f}")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.set_title(f"{ds} (r = {T.loc[('shipped, all windows', ds), 'r']:.2f})")
    ax.set_xlabel("ECG truth (BPM)")
    ax.legend(loc="upper left")
    put("scatter", **{f"slope_{ds}": slope, f"intercept_{ds}": icpt})
axes[0].set_ylabel("predicted (BPM)")
fig.colorbar(hb, ax=axes, label="windows (log)", shrink=0.8)
save(fig, "scatter")

## Bland-Altman

Difference (pred - truth) against the mean of both, as window density (log color; 414k multisite windows would
hide each other as points). Solid = bias, dashed = 95 % limits of agreement (bias ± 1.96 SD), window-pooled per sub-dataset.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=figsize(1.1), sharey=True)
BA = {}
for ax, ds in zip(axes, DATASETS):
    d = W[W["dataset"] == ds]
    diff, mean = d["kalman"] - d["truth"], (d["kalman"] + d["truth"]) / 2
    bias, sd = diff.mean(), diff.std()
    BA[ds] = {"bias": bias, "loa_low": bias - 1.96 * sd, "loa_high": bias + 1.96 * sd}
    hb = ax.hexbin(mean, diff, gridsize=60, extent=(35, 175, -60, 60), bins="log", cmap=DENSITY_CMAP, mincnt=1)
    ax.axhline(bias, color=COLORS["text"], lw=STYLE["lw"], label=f"bias {bias:+.2f}")
    for y in (BA[ds]["loa_low"], BA[ds]["loa_high"]):
        ax.axhline(y, color=COLORS["bad"], ls="--", lw=STYLE["lw_thin"])
    ax.plot([], [], color=COLORS["bad"], ls="--", label=f"LoA {BA[ds]['loa_low']:+.1f} / {BA[ds]['loa_high']:+.1f}")
    ax.set_ylim(-60, 60); ax.set_xlim(35, 175)
    ax.set_xlabel("mean of pred and truth (BPM)")
    ax.set_title(ds)
    ax.legend(loc="lower left")
axes[0].set_ylabel("pred - truth (BPM)")
fig.colorbar(hb, ax=axes, label="windows (log)", shrink=0.8)
fig.suptitle("Bland-Altman, shipped pipeline (all test windows)")
save(fig, "bland_altman")
put("bland_altman", **{f"{ds}_{k}": v for ds, b in BA.items() for k, v in b.items()})
pd.DataFrame(BA).T.round(2)

,bias,loa_low,loa_high
multisite-ppg,1.22,-19.91,22.34
wrist-finger-ppg,0.12,-4.26,4.50


## Error per BPM bin

Window-pooled MAE and bias per ECG truth bin, raw ensemble vs shipped pipeline. Numbers on top: windows per bin.
The high-BPM tail is rare in the data (2.8 % of windows >= 110 BPM) and stays the weak spot.

In [8]:
W["bin"] = bin_of(W["truth"])
g = W.assign(e_raw=(W["raw"] - W["truth"]), e_k=(W["kalman"] - W["truth"])).groupby("bin", observed=False)
B = pd.DataFrame({"n": g.size(), "mae_raw": g["e_raw"].apply(lambda e: e.abs().mean()),
                  "mae_shipped": g["e_k"].apply(lambda e: e.abs().mean()), "bias_shipped": g["e_k"].mean()})
fig, (a1, a2) = plt.subplots(2, 1, figsize=figsize(1.6), sharex=True)
x = np.arange(len(B))
a1.bar(x - 0.2, B["mae_raw"], 0.4, color=COLORS["neutral"], edgecolor=COLORS["muted"], label="raw ensemble")
a1.bar(x + 0.2, B["mae_shipped"], 0.4, color=COLORS[CFG], label="shipped (Kalman + input gate)")
for xi, (n, h) in enumerate(zip(B["n"], B[["mae_raw", "mae_shipped"]].max(1))):
    a1.annotate(f"{n}", (xi, h), xytext=(0, 3), textcoords="offset points", ha="center", color=COLORS["muted"])
a1.set_ylabel("MAE (BPM)"); a1.legend(loc="upper center")
a1.set_ylim(0, B[["mae_raw", "mae_shipped"]].max().max() * 1.2)
a2.bar(x, B["bias_shipped"], color=COLORS[CFG])
a2.axhline(0, color=COLORS["muted"], lw=STYLE["lw_thin"])
a2.set_ylabel("bias pred - truth (BPM)")
a2.set_xticks(x, BIN_LABELS, rotation=45); a2.set_xlabel("ECG truth (BPM bin)")
fig.suptitle("Error per heart-rate bin (test, all windows)")
save(fig, "bins")
put("bins", **{f"{m}_{b}": B.loc[b, m] for b in B.index for m in B.columns})
B.round(2)

,n,mae_raw,mae_shipped,bias_shipped
bin,,,,
40-60,39049,12.78,11.77,11.38
60-70,83489,9.14,7.85,7.62
70-80,113183,4.58,3.66,3.17
80-90,101011,3.11,2.19,-0.31
90-100,56346,6.59,5.83,-5.35
100-110,22214,13.43,12.49,-11.96
110-120,7149,20.94,19.94,-19.62
120-130,2733,30.04,28.33,-28.22
130+,2820,43.57,42.45,-42.37


## Per subject

MAE per test subject (shipped pipeline, all windows), sorted, colored by sub-dataset. Shows how much of the mean comes
from a few hard subjects.

In [9]:
S = (W.assign(err=(W["kalman"] - W["truth"]).abs()).groupby(["dataset", "subject"])
     .agg(mae=("err", "mean"), windows=("err", "size"), mean_hr=("truth", "mean")).reset_index().sort_values("mae"))
fig, ax = plt.subplots(figsize=figsize(0.9))
ax.bar(np.arange(len(S)), S["mae"], color=[COLORS[d] for d in S["dataset"]])
from matplotlib.patches import Patch
ax.set_xticks([]); ax.set_xlabel("test subjects, sorted")
ax.set_ylabel("MAE (BPM)")
ax.set_title("Per-subject MAE, out-of-fold test")
ax.legend(handles=[Patch(color=COLORS[ds], label=f"{ds} (median {S[S['dataset'] == ds]['mae'].median():.2f})")
                   for ds in DATASETS], loc="upper left")
save(fig, "subjects")
put("subjects", **{f"median_{ds}": S[S["dataset"] == ds]["mae"].median() for ds in DATASETS},
    **{f"max_{ds}": S[S["dataset"] == ds]["mae"].max() for ds in DATASETS})
S.tail(8).round(2)

,dataset,subject,mae,windows,mean_hr
2,multisite-ppg,multisite_P11,6.04,22265,81.19
18,multisite-ppg,multisite_P8,6.60,14858,94.02
1,multisite-ppg,multisite_P10,6.75,35709,93.13
13,multisite-ppg,multisite_P3,7.09,28254,93.15
11,multisite-ppg,multisite_P2,7.94,83408,68.15
9,multisite-ppg,multisite_P18,8.92,53046,65.76
8,multisite-ppg,multisite_P17,10.35,13273,97.97
5,multisite-ppg,multisite_P14,12.36,13541,62.72


## Error distribution

Cumulative share of windows with |error| <= x. Read off: share within 5 / 10 BPM.

In [10]:
fig, ax = plt.subplots(figsize=figsize(0.85))
for ds in DATASETS:
    d = W[W["dataset"] == ds]
    for col, ls, lab in [("raw", "--", "raw ensemble"), ("kalman", "-", "shipped")]:
        e = np.sort((d[col] - d["truth"]).abs().to_numpy())
        ax.plot(e, np.arange(1, len(e) + 1) / len(e), color=COLORS[ds], ls=ls, label=f"{ds}, {lab}")
for x in (5, 10):
    ax.axvline(x, color=COLORS["muted"], ls=":", lw=STYLE["lw_thin"])
ax.set_xlim(0, 30); ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("|pred - truth| (BPM)"); ax.set_ylabel("share of windows")
ax.set_title("Cumulative absolute error (test)")
ax.legend(loc="lower right")
save(fig, "error_cdf")

## Train vs test: the 5 fold ensembles

Each fold's 3-seed ensemble (raw, no gate, no smoothing) scored on its own **train** subjects, its **val** group and
its **test** group. Windows: the real labelled windows of `cache/windows/` (the same capped set training sampled
from; no bootstrap windows, no noise augmentation). So train MAE = error on data the model was fitted on; the
train-to-test gap measures over-fitting. Metric: subject-macro MAE per sub-dataset.

In [11]:
from bpm.data_loading import load_subject_npz
from evaluate import load_model, predict

SPLITS = ["train", "val", "test"]
rows = []
for k in range(5):
    sp = fold_split(groups, k)
    models = [load_model(ROOT / "cache" / f"best_{CFG}_fold{k}_seed{s}.pt") for s in (42, 1, 2)]
    for split_name in SPLITS:
        for subj in sorted(sp[split_name]):
            d = load_subject_npz(subj, ROOT / "cache" / "windows")
            if d is None or not len(d["labels"]):
                continue
            pred = predict(models, d)["pred"]
            rows.append({"fold": k, "split": split_name, "subject": subj, "dataset": str(d["dataset_source"][0]),
                         "mae": np.abs(pred - d["labels"]).mean(), "windows": len(pred)})
TT = pd.DataFrame(rows)
# check: test/val MAE per subject equals the Stage D preds file of the same ensemble
for k in range(5):
    z = np.load(PREDS / f"ens_{CFG}_fold{k}.npz")
    ref = pd.DataFrame({"subject": z["subject"], "split": z["split"], "err": np.abs(z["pred"] - z["truth"])}) \
        .groupby(["split", "subject"])["err"].mean()
    mine = TT[(TT["fold"] == k) & (TT["split"] != "train")].set_index(["split", "subject"])["mae"]
    assert np.allclose(mine.sort_index(), ref.sort_index(), atol=1e-3), k
print(TT.groupby("split")["subject"].count().to_dict(), "subject-fold rows;", TT.groupby("split")["windows"].sum().to_dict(), "windows")

{'test': 47, 'train': 141, 'val': 47} subject-fold rows; {'test': 51864, 'train': 155592, 'val': 51864} windows


In [12]:
FOLDS = TT.groupby(["dataset", "fold", "split"])["mae"].mean().unstack("split")[SPLITS]
SUMMARY = FOLDS.groupby("dataset").agg(["mean", "std"])
both = FOLDS.groupby("fold").mean()  # mean of the 2 sub-datasets per fold
SUMMARY.loc["both"] = both.agg(["mean", "std"]).T.stack()
FOLDS.to_csv(RESULTS / f"final_{CFG}_train_test.csv")
put("train_test", **{f"{ds}_{s}_{m}": SUMMARY.loc[ds, (s, m)] for ds in SUMMARY.index for s in SPLITS for m in ("mean", "std")})
display(FOLDS.round(2))
SUMMARY.round(2)

split                  train   val  test
dataset          fold                   
multisite-ppg    0      5.82  5.73  6.91
                 1      5.39  6.62  6.01
                 2      6.08  6.02  7.48
                 3      6.30  7.66  6.63
                 4      4.95  6.64  7.37
wrist-finger-ppg 0      1.06  1.49  1.45
                 1      1.12  0.80  1.53
                 2      1.09  2.16  0.75
                 3      1.08  1.41  2.25
                 4      1.05  1.52  1.41

split            train         val        test      
                  mean   std  mean   std  mean   std
dataset                                             
multisite-ppg     5.71  0.54  6.53  0.74  6.88  0.60
wrist-finger-ppg  1.08  0.03  1.48  0.48  1.48  0.53
both              3.39  0.27  4.00  0.37  4.18  0.27

In [13]:
MK_SPLIT = {"train": "o", "val": "o", "test": "D"}
FILL = {"train": True, "val": False, "test": True}
fig, axes = plt.subplots(1, 2, figsize=figsize(0.95))
for ax, ds in zip(axes, DATASETS):
    f = FOLDS.loc[ds]
    ylab = [f"fold {k}" for k in f.index] + ["mean ± sd"]
    y = np.arange(len(ylab))[::-1]
    for yi, (_, r) in zip(y, f.iterrows()):
        ax.plot([r.min(), r.max()], [yi, yi], color=COLORS["neutral"], lw=3, zorder=1)
    m, sd = SUMMARY.loc[ds].xs("mean", level=1), SUMMARY.loc[ds].xs("std", level=1)
    ax.plot([m.min(), m.max()], [y[-1], y[-1]], color=COLORS["neutral"], lw=3, zorder=1)
    for s in SPLITS:
        c = COLORS[s]
        ax.scatter(f[s], y[:-1], s=STYLE["marker"], marker=MK_SPLIT[s], color=c if FILL[s] else COLORS["surface"],
                   edgecolor=c, linewidth=1.5, zorder=3, label=s)
        ax.errorbar(m[s], y[-1], xerr=sd[s], fmt=MK_SPLIT[s], ms=8, color=c, mfc=c if FILL[s] else COLORS["surface"],
                    mec=c, mew=1.5, capsize=3, zorder=3)
    ax.axhline(y[-1] + 0.5, color=COLORS["grid"], lw=STYLE["lw_thin"])
    ax.set_yticks(y, ylab)
    ax.set_xlim(0, None)
    ax.set_xlabel("MAE (BPM), subject-macro")
    ax.set_title(f"{ds}: train {m['train']:.2f} vs test {m['test']:.2f}")
axes[0].legend(loc="upper left")
fig.suptitle("dist + boot, 3-seed ensemble per fold: train vs val vs test")
save(fig, "train_test_folds")

One dot per subject. Train: mean over the 3 folds in which the subject was in train; val and test: the one fold each.
Thin lines join the same subject across splits.

In [14]:
PS = TT.groupby(["dataset", "subject", "split"])["mae"].mean().unstack("split")[SPLITS]
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=figsize(1))
for ax, ds in zip(axes, DATASETS):
    p = PS.loc[ds]
    jit = rng.uniform(-0.12, 0.12, len(p))
    ax.plot(np.arange(3)[:, None] + jit[None, :], p[SPLITS].T.to_numpy(), color=COLORS["neutral"], lw=0.6, zorder=1)
    bp = ax.boxplot([p[s] for s in SPLITS], positions=range(3), widths=0.5, showfliers=False, patch_artist=True,
                    medianprops={"color": COLORS["text"], "lw": STYLE["lw"]},
                    boxprops={"facecolor": COLORS["surface"], "edgecolor": COLORS["muted"]},
                    whiskerprops={"color": COLORS["muted"]}, capprops={"color": COLORS["muted"]}, zorder=2)
    for i, s in enumerate(SPLITS):
        ax.scatter(i + jit, p[s], s=STYLE["marker"] * 0.5, color=COLORS[s], alpha=0.85, zorder=3, lw=0)
    ax.set_xticks(range(3), [f"{s}\nmedian {p[s].median():.2f}" for s in SPLITS])
    ax.set_ylim(0, None)
    ax.set_ylabel("MAE per subject (BPM)")
    ax.set_title(f"{ds} ({len(p)} subjects)")
fig.suptitle("Per-subject MAE: train vs val vs test (raw 3-seed ensembles)")
save(fig, "train_test_subjects")
put("train_test", **{f"{ds}_median_{s}": PS.loc[ds][s].median() for ds in DATASETS for s in SPLITS})
PS.groupby("dataset").median().round(2)

split,train,val,test
dataset,,,
multisite-ppg,4.82,6.03,6.28
wrist-finger-ppg,0.85,1.03,1.18


In [15]:
print(json.dumps(VALUES, indent=1, default=float)[:3000])

{
 "table": {
  "raw ensemble|multisite-ppg|coverage": 1.0,
  "raw ensemble|multisite-ppg|subjects": 20,
  "raw ensemble|multisite-ppg|windows": 413798,
  "raw ensemble|multisite-ppg|mae_subject": 6.734999305380077,
  "raw ensemble|multisite-ppg|mae_window": 7.4869887411206335,
  "raw ensemble|multisite-ppg|rmse": 13.350479381124831,
  "raw ensemble|multisite-ppg|bias": 1.8060674276161055,
  "raw ensemble|multisite-ppg|r": 0.5739857773398702,
  "raw ensemble|multisite-ppg|within_5": 0.6191040072692473,
  "raw ensemble|multisite-ppg|within_10": 0.744735353964978,
  "raw ensemble|wrist-finger-ppg|coverage": 1.0,
  "raw ensemble|wrist-finger-ppg|subjects": 27,
  "raw ensemble|wrist-finger-ppg|windows": 14196,
  "raw ensemble|wrist-finger-ppg|mae_subject": 1.477425700521293,
  "raw ensemble|wrist-finger-ppg|mae_window": 1.4984929806818656,
  "raw ensemble|wrist-finger-ppg|rmse": 2.5310266450186174,
  "raw ensemble|wrist-finger-ppg|bias": 0.16491350854877418,
  "raw ensemble|wrist-finger-pp